In [ ]:
import torch
from torch import nn
import torch.optim as optim
import torch.nn.functional as F
from transformers import T5ForConditionalGeneration, T5Tokenizer
import numpy as np
import datasets
from datasets import DatasetDict
from sklearn.model_selection import train_test_split
import random

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt


In [ ]:
# fix seeds for reproducibility
#torch.manual_seed(0)
#random.seed(0)
#np.random.seed(0)

In [2]:
model_name = 'jbochi/madlad400-3b-mt'
#model = T5ForConditionalGeneration.from_pretrained(model_name, device_map=None)
tokenizer = T5Tokenizer.from_pretrained(model_name)

In [50]:
print(f"> Number of parameters in the model: {sum(p.numel() for p in model.parameters()):,}")


NameError: name 'model' is not defined

In [ ]:
gatitos: DatasetDict = datasets.load_dataset("google/smol", "gatitos__yue_zh") # type: ignore
smolsent: DatasetDict = datasets.load_dataset("google/smol", "smolsent__en_yue") # type: ignore
smoldoc: DatasetDict = datasets.load_dataset("google/smol", "smoldoc__en_yue") # type: ignore

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


Define Hyperparameters

In [7]:
epochs = 5
batch_size = 16
max_len = 256
learning_rate = 1e-4

Create dataloaders

In [ ]:
class SmolsentDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
    
    def __len__(self):
        return self.dataset.__len__()
    
    def __getitem__(self, idx):
        return self.dataset[idx]["trg"], self.dataset[idx]["src"]



smolsent_train_validation = smolsent["train"].train_test_split(train_size=0.9)
smolsent_train_test = smolsent_train_validation["train"].train_test_split(train_size=8/9)

smolsent_train = smolsent_train_test["train"]
smolsent_validation = smolsent_train_validation["test"]
smolsent_test = smolsent_train_test["train"]

smolsent_train_dataloader = DataLoader(SmolsentDataset(smolsent_train), batch_size=batch_size, shuffle=True)
smolsent_validation_dataloader = DataLoader(SmolsentDataset(smolsent_validation), batch_size=batch_size, shuffle=True)
smolsent_test_dataloader = DataLoader(SmolsentDataset(smolsent_test), batch_size=batch_size, shuffle=True)


In [48]:
for x in smolsent_train_dataloader:
    print(x[0])
    print(x[1])
    print(len(x[0]))
    break

('三月係巴巴多斯嘅旅遊旺季，所以可以預期有怡人、溫暖同陽光普照嘅天氣。', '如果我同1940年比較，雖然講到明禁止，但我只係想肯定我冇做出錯誤嘅假設。', '兩個開心嘅後生女喺屋企度開派對開到好夜，又笑，又飲啤酒食薄餅。', '著西裝可以減少其他人對你嘅偏見。', '我哋嘅口碑建立喺出色嘅客戶服務上，同時我哋亦都致力幫你處理你嘅稅務問題。', '當你犯錯嘅時候，大海會即刻提醒你。', '要小心評估雪地嘅狀況，謹慎咁選擇路線同埋保守咁做決定係至關重要。', '冰霜同雷聲，火光烈烈，大地今晚就要步向毀滅。', '呢個等同於四級颶風嘅風力！', '商業預支現金機構對於風險評估同計算信用要求嘅方式同傳統銀行可能有啲唔同。', '願上帝喺你嘅生日同埋往後嘅日子，賜予你快樂、滿足、繁榮、和平同豐盛。', '當我哋係服從、忠誠，同埋熱誠而誠實嘅時候，每次我哋向上帝禱告，奇蹟都會發生。', '籌備完考試之後，委員會仲會進行面試，招募有能之士擔任相關職位。', '你真係要學返點樣食早餐、午餐同晚餐。', '今日不妨可以盡力為每一餐感恩。', '獎勵機制唔好，經理想點就點又無耐性，所以我同事辭咗職。')
("March falls in Barbados' peak tourist season so pleasant, warm and sunny weather can be expected.", 'If I compare with 1940, where it is specifically forbidden, I simply want to make sure I am not making a false assumption.', 'Two happy young girls laughing, drinking beer, and eating pizza at a home party late.', 'Wear a suit to decrease the bias others will experience towards you.', 'We have built our reputation on outstanding client service and are committed to helping you succeed in your 

Training Loop

In [49]:
num_batches = int(np.ceil(len(smolsent_train_dataloader) / batch_size))

loss_fn = nn.CrossEntropyLoss(ignore_index=hf_tokenizer.pad_token_id)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate)

NameError: name 'hf_tokenizer' is not defined

In [ ]:
train_losses = []

model.to(device)
model.train() 
for epoch_num in range(epochs):
    epoch_loss = []
    for inputs, targets in smolsent_train_dataloader: 
        src_tokens = tokenizer(inputs, return_tensors='pt', padding=True, truncation= True, max_length=max_len)
        tgt_tokens = tokenizer(targets, return_tensors='pt', padding=True, truncation= True, max_length=max_len)
        
        logits, _ = model(
            src=src_tokens["input_ids"],
            tgt=tgt_tokens["input_ids"],
            src_mask=src_tokens["attention_mask"],
            tgt_mask=tgt_tokens["attention_mask"],
        )

        labels = tgt_tokens["input_ids"][:, 1:].contiguous()
        logits = logits[:, :-1].contiguous() 

        optimizer.zero_grad()
        loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
        loss.backward()
        loss.step()

        print(f"Epoch {epoch_num:2d}, Batch loss: {loss.item():.3f}", end='\r')
        epoch_loss.append(loss.item())

print(f"Epoch {epoch_num:2d}, Loss: {np.mean(epoch_loss):.4f}")
train_losses.append(np.mean(epoch_loss))


In [ ]:
plt.plot(train_losses)